In [2]:
import csv
import tiktoken
import pandas as pd
import os
from tqdm.auto import tqdm
import json

In [3]:
ann1 = pd.read_csv('../agreed_annotations.csv')
ann1_filtered = ann1[ann1.Valid.str.strip().isin(['Y', 'N', 'y', 'n'])]
ann1_filtered.rename(columns={'Dialogue': 'dialogue', 'Cluster': 'theme', 'Norm': 'norm', 'Summary': 'summary', 'Valid': 'applicability'}, inplace=True)
ann1_filtered = ann1_filtered[['dialogue', 'theme', 'norm', 'summary', 'applicability']]
ann1_filtered['applicability'] = ann1_filtered['applicability'].apply(lambda x:x.lower() == 'y')
print(len(ann1), len(ann1_filtered))

172 172


In [20]:
ann1_filtered.dialogue.nunique()

160

In [4]:
# Convert phase 2 annotations to standard format

ann2 = pd.read_csv('../additional_annotations_norms_v5.csv')
ann2.rename(columns={'display_string': 'dialogue', 'theme_name': 'theme', 'norm_text': 'norm', 'applicable?': 'applicability'}, inplace=True)
ann2['dialogue'] = ann2['dialogue'].apply(lambda x:str(x).replace('<b>', '').replace('</b>', '').replace('<br>', '\n').strip())
ann2_filtered = ann2[['dialogue_id', 'dialogue', 'theme', 'norm', 'summary', 'applicability']]
ann2_dialogues = ann2_filtered[['dialogue_id', 'dialogue', 'summary']]
ann2_dialogues = ann2_dialogues[~ann2_dialogues.dialogue.str.strip().isin(['nan'])]
ann2_filtered_dialogues = pd.merge(ann2_filtered, ann2_dialogues, on='dialogue_id', how='inner')
ann2_filtered_dialogues = ann2_filtered_dialogues[['dialogue_y', 'theme', 'norm', 'summary_y', 'applicability']]
ann2_filtered_dialogues.rename(columns={'dialogue_y': 'dialogue', 'summary_y': 'summary'}, inplace=True)
ann2_filtered_dialogues[:3]
ann2_filtered_dialogues = ann2_filtered_dialogues[ann2_filtered_dialogues.applicability.str.upper().isin(['TRUE', 'FALSE'])]
ann2_filtered_dialogues['applicability'] = ann2_filtered_dialogues['applicability'].apply(lambda x:x.upper() == 'TRUE')

print(len(ann2), len(ann2_filtered), len(ann2_filtered_dialogues))

1635 1635 726


In [5]:
ann2_appl = ann2[ann2.applicability.str.upper().isin(['TRUE', 'FALSE'])]
print(len(ann2_appl))

726


In [13]:
ann2_appl.applicability.value_counts()

applicability
TRUE     588
FALSE    138
Name: count, dtype: int64

In [14]:
588/726

0.8099173553719008

In [6]:
verification_output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_verification_outputs/'
bnames = os.listdir(verification_output_dir)
resps = {}
outs = {}
full_outs = {}
for bname in tqdm(bnames):
    bres = json.load(open(os.path.join(verification_output_dir, bname)))
    for out in bres:
        n_id, resp = out
        full_outs[n_id] = resp.strip()
        resp = resp.strip().split('Relevance:')[1].strip().lower()
        if resp not in resps:
            resps[resp] = 1
        else:
            resps[resp] += 1
        if resp in ['relevant', 'irrelevant']:
            outs[n_id] = resp 
        elif n_id in [225, 22829]:
            outs[n_id] = 'relevant' 
            
# display(resps)
print(len(outs))

  0%|          | 0/64 [00:00<?, ?it/s]

63754


In [7]:
agenteval_output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/relevance_agenteval_outputs/'
bnames = os.listdir(agenteval_output_dir)
agenteval_resps = {}
agenteval_outs = {}
agenteval_full_outs = {}
agenteval_metrics_outs = {}
w = 0

w_bnames = set()

for bname in tqdm(bnames):
    bres = json.load(open(os.path.join(agenteval_output_dir, bname)))
    for out in bres:
        # print(out)
        n_id, _, metrics, resp = out
        
        if 'relevance:' in resp.lower(): 
            agenteval_full_outs[n_id] = resp.strip()
            resp = resp.strip().lower().split('relevance:')[1].strip()
            try:   
                agenteval_metrics_outs[n_id] = json.loads(metrics.strip())
            except:
                agenteval_metrics_outs[n_id] = {}
        else:
            agenteval_full_outs[n_id] = metrics.strip()
            resp = metrics.strip().lower().split('relevance:')[1].strip()
            try:   
                agenteval_metrics_outs[n_id] = json.loads(resp.strip())
            except:
                agenteval_metrics_outs[n_id] = {}
        if resp not in agenteval_resps:
            agenteval_resps[resp] = 1
        else:
            agenteval_resps[resp] += 1
        if resp in ['relevant', 'irrelevant']:
            agenteval_outs[n_id] = resp
        elif resp in ['highly relevant', 'highly appropriate']:
            agenteval_outs[n_id] = 'relevant'
        elif resp in ['minimally appropriate']:
            agenteval_outs[n_id] = 'irrelevant'
        elif resp in ['somewhat relevant', 'somewhat appropriate', '3 - somewhat appropriate']:
            agenteval_outs[n_id] = 'semi-relevant'
        else:
            print(resp)
            w += 1
        # elif n_id in [225, 22829]:
        #     agenteval_outs[n_id] = 'relevant' 
            
# display(resps)
print(w)
print(len(agenteval_outs))

  0%|          | 0/64 [00:00<?, ?it/s]

0
63779


In [8]:
norm_prompts = json.load(open('/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_prompts.json'))[1]

In [9]:
i = 2
fw = open(f'./successful_relevance_{i}.txt', 'w')
fw.write(f"### Input:\n\n{norm_prompts['9'].strip()}\n\n### Response:\n\n{full_outs[9].strip()}")
fw.close()

In [10]:
task_prompt = json.load(open('/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_prompts.json'))[0]

fw = open(f'./relevance_prompt.txt', 'w')
fw.write(task_prompt.strip())
fw.close()

In [11]:
tp, t = 0, 0
tpq, tq = 0, 0

sel_nids = set()
for index, row in ann2_appl.iterrows():
    n_id = row['norm_id']
    if n_id in outs:
        appl = row['applicability']
        r = outs[n_id]
        if appl == 'TRUE':
            tp += 1
        t += 1
        if r == 'relevant':
            sel_nids.add(n_id)
            if appl == 'TRUE':
                tpq += 1
            tq += 1
print(tpq/tq, tq)

0.8767605633802817 568


In [15]:
tpq/588

0.8469387755102041

In [16]:
tp, t = 0, 0
tpq, tq = 0, 0

for index, row in ann2_appl.iterrows():
    n_id = row['norm_id']
    if n_id in agenteval_outs and n_id in sel_nids:
        appl = row['applicability']
        r = agenteval_outs[n_id]
        if appl == 'TRUE':
            tp += 1
        t += 1
        if r == 'relevant':
            if appl == 'TRUE':
                tpq += 1
            tq += 1
        elif r == 'semi-relevant':
            tpq += 1
            tq += 1
            
print(tpq/tq, tq)

tp, t = 0, 0
tpq, tq = 0, 0

for index, row in ann2_appl.iterrows():
    n_id = row['norm_id']
    if n_id in agenteval_outs:
        appl = row['applicability']
        r = agenteval_outs[n_id]
        if appl == 'TRUE':
            tp += 1
        t += 1
        if r == 'relevant':
            if appl == 'TRUE':
                tpq += 1
            tq += 1
        elif r == 'semi-relevant' and appl == 'TRUE':
            tpq += 1
            tq += 1
            
print(tpq/tq, tq)

0.8840579710144928 552
0.8550955414012739 628


In [17]:
tpq/588

0.9132653061224489

In [1]:
553/726

0.7617079889807162

In [10]:
print(len(ann2_appl))

726


In [9]:
#applicability
print(tp, t, tp/t)
print(tpq, tq, tpq/tq)

588 726 0.8099173553719008
498 568 0.8767605633802817


In [118]:
all_data_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_dialogues_themes_data.csv')

In [119]:
all_data_df[:3]

,Unnamed: 0,id__x,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,...,other_features,dialogue_text_zh,id_,name,description,violation_characteristic,activation_settings,actors,recepients,symbolic_prompt
0,0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent,Norm ID: 0\nTheme ID: 16\n\nConversation from ...
1,1,15,2. Filial piety,0.2567,False,False,[],[],24,24,...,{},徐父(angry): 是嗎，舊的不爛，新的不會來嘛！我還要繼續砸呢！\n徐父(angry):...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent,Norm ID: 15\nTheme ID: 16\n\nConversation from...
2,2,18,2. Filial piety: Children are expected to prio...,0.0858,False,False,[],[],24,24,...,{},左母(surprise): 正鵬，今天你到那個憨同學家裏，她父母跟你說了些什麼？她自己呢？\...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent,Norm ID: 18\nTheme ID: 16\n\nConversation from...


In [120]:
print(all_data_df.columns)

Index(['Unnamed: 0', 'id__x', 'text', 'distance', 'good', 'validated',
       'gpt3_answer_rating', 'gpt3_feedback', 'actor_role_id',
       'recepient_role_id', 'dialogue_id', 'theme_id', 'id__y', 'identifier',
       'culture', 'dialogue_string', 'display_text', 'summary',
       'other_features', 'dialogue_text_zh', 'id_', 'name', 'description',
       'violation_characteristic', 'activation_settings', 'actors',
       'recepients', 'symbolic_prompt'],
      dtype='object')


In [121]:
print(agenteval_full_outs[10])

Justification: The social norm of filial piety is highly relevant in this conversation as it reflects the cultural expectation of hospitality and care for guests, particularly in the context of family relationships. Mrs. Xu's instruction to her husband to buy alcohol for their daughter's friend demonstrates a strong adherence to this norm, showcasing their desire to provide for and welcome the guest. The setting is familial, and the relationship dynamics are clearly supportive, further emphasizing the appropriateness of the norm. Additionally, the emotional tone is positive, and the actions align well with established cultural practices, making the norm both visible and widely adopted in Chinese society. Overall, the judgment considers various contextual factors, including the relationships involved and the cultural significance of the actions taken. 
Relevance: relevant


In [122]:
all_data_df['relevance_judgment'] = all_data_df['id__x'].map(agenteval_outs)
all_data_df['relevance_justification'] = all_data_df['id__x'].map(agenteval_full_outs)
all_data_df['relevance_metrics'] = all_data_df['id__x'].map(agenteval_metrics_outs)

In [123]:
print(all_data_df.columns)

Index(['Unnamed: 0', 'id__x', 'text', 'distance', 'good', 'validated',
       'gpt3_answer_rating', 'gpt3_feedback', 'actor_role_id',
       'recepient_role_id', 'dialogue_id', 'theme_id', 'id__y', 'identifier',
       'culture', 'dialogue_string', 'display_text', 'summary',
       'other_features', 'dialogue_text_zh', 'id_', 'name', 'description',
       'violation_characteristic', 'activation_settings', 'actors',
       'recepients', 'symbolic_prompt', 'relevance_judgment',
       'relevance_justification', 'relevance_metrics'],
      dtype='object')


In [124]:
all_data_df.to_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_dialogues_themes_relevance_data.csv')